In [ ]:
!pip install math_verify num2words==0.5.14 peft
!pip install -q trl peft math-verify latex2sympy2_extended datasets accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/Qwen3-0.6B-RLOO-NuminaMath"

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

import torch
from datasets import load_dataset



# Dataset
train_dataset, eval_dataset = load_dataset("AI-MO/NuminaMath-TIR", split=["train[:5%]", "test[:50%]"])
train_dataset = train_dataset.select(range(500))
eval_dataset = eval_dataset.select(range(50))

SYSTEM_PROMPT = (
    "A conversation between user and assistant. The user asks a question, and the assistant solves it. The "
    "assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
    "The reasoning process and answer are enclosed within <think></think> tags, i.e., <think>\nThis is my "
    "reasoning.\n</think>\nThis is my answer."
)

def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

train_dataset = train_dataset.map(make_conversation, remove_columns=["messages", "problem"])
eval_dataset = eval_dataset.map(make_conversation, remove_columns=["messages", "problem"])

In [ ]:
print("Prompt:\n", eval_dataset[5]['prompt'][0], "\n", eval_dataset[5]['prompt'][1])
print("\nSolution:\n", eval_dataset[5]['solution'])

Fine-tuning

In [ ]:
from peft import LoraConfig

from trl import RLOOConfig, RLOOTrainer
from trl.rewards import accuracy_reward, think_format_reward

# Training
training_args = RLOOConfig(
    output_dir=OUTPUT_DIR,
    learning_rate=1e-5,
    log_completions=True,
    num_completions_to_print=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_completion_length=512,
    steps_per_generation=4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

trainer = RLOOTrainer(
    model="Qwen/Qwen3-0.6B",
    args=training_args,
    reward_funcs=[think_format_reward, accuracy_reward],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
print("Modello salvato in:", OUTPUT_DIR)


Eval

In [ ]:
import torch
import re
from fractions import Fraction
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from trl.rewards import think_format_reward, accuracy_reward

# CONFIG
REFERENCE_MODEL_NAME = "Qwen/Qwen3-0.6B"
POLICY_MODEL_PATH = OUTPUT_DIR
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 1024


# LOAD TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(REFERENCE_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token


# LOAD MODELS
def load_reference():
    return AutoModelForCausalLM.from_pretrained(
        REFERENCE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        tie_word_embeddings=False
    ).eval()

def load_policy():
    base = AutoModelForCausalLM.from_pretrained(
        REFERENCE_MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        tie_word_embeddings=False
    )
    return PeftModel.from_pretrained(base, POLICY_MODEL_PATH).eval()


# FORMAT PROMPT
def format_chat_prompt(sample):
  return tokenizer.apply_chat_template(
      sample,
      tokenize=False,
      add_generation_prompt=True, )


# GENERATION
def generate_batch(model, prompts):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
        )

    decoded_texts = []
    for i, prompt in enumerate(prompts):
        input_len = inputs["input_ids"][i].shape[0]
        generated_tokens = outputs[i][input_len:]
        decoded = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )
        decoded_texts.append(decoded)
    return decoded_texts


# BATCH EVALUATION (mean reward)
def evaluate_model(model, dataset):
    total_reward = 0
    n = len(dataset)

    for i in tqdm(range(0, n, BATCH_SIZE)):
        batch = dataset[i:i+BATCH_SIZE]
        prompts = [format_chat_prompt(s) for s in batch["prompt"]]

        outputs = generate_batch(model, prompts)


        # wrap in TRL completions format
        completions = [[{"content": out}] for out in outputs]
        solutions = [sol for sol in batch["solution"]]


        r1 = think_format_reward(completions)
        r2 = accuracy_reward(completions, solutions)
        total_reward = sum(r1) + sum(r2)

    mean_reward = total_reward / n
    return mean_reward


# RUN EVALUATION
reference_model = load_reference()
policy_model = load_policy()

print("Evaluating Reference Model...")
ref_reward = evaluate_model(reference_model, eval_dataset)

print("Evaluating Policy Model...")
policy_reward = evaluate_model(policy_model, eval_dataset)

print("\n===== RESULTS =====")
print(f"Reference Model Mean Reward: {ref_reward:.4f}")
print(f"Policy Model    Mean Reward: {policy_reward:.4f}")